To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [6]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [32]:
from unsloth import FastSentenceTransformer

fourbit_models = [
    "unsloth/all-MiniLM-L6-v2",
    "unsloth/embeddinggemma-300m",
    "unsloth/Qwen3-Embedding-4B",
    "unsloth/Qwen3-Embedding-0.6B",
    "unsloth/all-mpnet-base-v2",
    "unsloth/gte-modernbert-base",
    "unsloth/bge-m3"

] # More models at https://huggingface.co/unsloth

original_model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/bge-m3",
    max_seq_length = 512,   # Choose any for long context!
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

Unsloth: Not planning a device map; SentenceTransformer moves the assembled model onto a single device. Using `sequential`.
Unsloth: Using fast encoder path for xlm-roberta (torch.compile + SDPA)


We now add LoRA adapters so we only need to update a small amount of parameters!

In [153]:
#model = FastSentenceTransformer.get_peft_model(
 #   model,
  #  r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
   # target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
     #                 "gate_proj", "up_proj", "down_proj",],
    #lora_alpha = 32,
    #lora_dropout = 0, # Supports any, but = 0 is optimized
    #bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    ##use_gradient_checkpointing = False, # True or "unsloth" for very long context
    #random_state = 3407,
    #use_rslora = False,  # We support rank stabilized LoRA
    #loftq_config = None, # And LoftQ
    #task_type = "FEATURE_EXTRACTION"
#)
model = FastSentenceTransformer.get_peft_model(
    original_model,

    r = 32,

    target_modules = [
        "query",
        "key",
        "value",
        "dense",
    ],

    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = False,

    random_state = 3407,

    use_rslora = False,
    loftq_config = None,

    task_type = "FEATURE_EXTRACTION",
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Unsloth: torch.compile will be applied automatically if max_steps > 242


<a name="Data"></a>
### Data Prep
We now use the `electroglyph/technical` dataset.

In [154]:
import pandas as pd
from datasets import Dataset

# Load the jsonl file into a pandas DataFrame, specifying lines=True for jsonl format
df = pd.read_json("/content/dataset_new.jsonl", lines=True)

# Fill None values in text columns with empty strings to prevent ValueError during training
df['anchor'] = df['anchor'].fillna('')
df['positive'] = df['positive'].fillna('')
df['negative'] = df['negative'].fillna('')

# Convert the pandas DataFrame to a datasets.Dataset object
dataset = Dataset.from_pandas(df)

Let's take a look at the dataset structure:

In [155]:
print("Dataset examples:")
for i in range(6):
    print(dataset[i])

Dataset examples:
{'anchor': 'Python', 'positive': 'Python is a programming language.', 'negative': ''}
{'anchor': 'Python', 'positive': 'Python is commonly used for automation and data analysis.', 'negative': ''}
{'anchor': 'Qt', 'positive': 'Qt is a cross-platform application framework.', 'negative': ''}
{'anchor': 'Qt', 'positive': 'Qt provides tools for developing graphical user interfaces.', 'negative': ''}
{'anchor': 'PodCertificateRequest', 'positive': 'Kubernetes provides native Pod certificate support through PodCertificateRequest.', 'negative': ''}
{'anchor': 'PodCertificateRequest', 'positive': 'PodCertificateRequest is used to request workload certificates in Kubernetes.', 'negative': ''}


## Baseline Inference
Let's test the base model before fine-tuning to see how it performs on our specific domain.

In [156]:
from sentence_transformers import util
import torch

def test_inference(model, run_name = "Run"):
    """Test model with a query and candidate sentences"""
    query = "PodCertificateRequest"
    candidates = [
                "a brick left by Yuki",  # Completely unrelated
        "apples are a tasty treat",  # Unrelated, but shares "ap-" prefix
        "A gentic modified plant",  # Unrelated
        "a type of cancer treatment that uses drugs to boost the body's immune response",  # Medical context but wrong procedure
        "a plant hormone for regulating stress responses",  # Scientific but unrelated field
        "induces root tip closure in non-vital teeth",  # CORRECT - this is what apexification actually means
        "Cas Crop Genome Editing",
        "It is used by Kubernetes v1.35 native Pod certificate support, allowing workload certificates to be issued and rotated without relying entirely on external certificate controllers",
        "Configuration data for kubernets work loads",
        "It is used by Podman  native Pod certificate support, allowing workload certificates to be issued and rotated without relying entirely on external certificate controllers.",
        "popular rifle cartridge used for hunting and target shooting"

    ]

    with torch.inference_mode():
      with torch.autocast(device_type = "cuda", dtype = torch.float32):
          query_emb = model.encode(query, convert_to_tensor = True)
          candidate_embs = model.encode(candidates, convert_to_tensor = True)
    scores = util.cos_sim(query_emb, candidate_embs)[0]

    results = []
    for i, score in enumerate(scores):
        results.append((candidates[i], score.item()))
    results.sort(key = lambda x: x[1], reverse = True)

    print(f"\n--- {run_name} Results for query: '{query}' ---")
    for text, score in results:
        print(f"{score:.4f} | {text}")

test_inference(model, run_name = "Pre-Training")


--- Pre-Training Results for query: 'PodCertificateRequest' ---
0.5980 | It is used by Podman  native Pod certificate support, allowing workload certificates to be issued and rotated without relying entirely on external certificate controllers.
0.5899 | It is used by Kubernetes v1.35 native Pod certificate support, allowing workload certificates to be issued and rotated without relying entirely on external certificate controllers
0.4158 | apples are a tasty treat
0.4043 | Cas Crop Genome Editing
0.3552 | Configuration data for kubernets work loads
0.3544 | A gentic modified plant
0.3253 | induces root tip closure in non-vital teeth
0.3208 | a plant hormone for regulating stress responses
0.2841 | a type of cancer treatment that uses drugs to boost the body's immune response
0.2755 | a brick left by Yuki
0.2639 | popular rifle cartridge used for hunting and target shooting


<a name="Train"></a>
### Train the model
Now let's train our model. We use `MultipleNegativesRankingLoss`

 This loss function uses other positives in the same batch as negative examples, which is efficient for contrastive learning.

In [157]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported

# This will use other positives in the same batch as negative examples
loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = dataset,
    loss = loss,
    args = SentenceTransformerTrainingArguments(
        output_dir = "output",
        num_train_epochs = 10,
        per_device_train_batch_size = 20,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        learning_rate = 3e-3,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        warmup_ratio = 0.03,
        report_to = "none",
        lr_scheduler_type = "constant_with_warmup",
        # Because we have duplicate anchors in the dataset, we don't want
        # to accidentally use them for negative examples
        batch_sampler = BatchSamplers.NO_DUPLICATES,
    ),

)

In [158]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
3.121 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [159]:
trainer_stats = trainer.train()

Step,Training Loss
1,0.151900
2,0.026100
3,0.000500
4,0.000000
5,0.138700
6,0.857300
7,0.461700
8,0.718800
9,3.679400
10,0.002500


In [160]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

24.1245 seconds used for training.
0.4 minutes used for training.
Peak reserved memory = 3.121 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 21.431 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model after training to see the improvements!

In [161]:
test_inference(model, run_name = "Post-Training")


--- Post-Training Results for query: 'PodCertificateRequest' ---
1.0000 | A gentic modified plant
1.0000 | a brick left by Yuki
1.0000 | apples are a tasty treat
1.0000 | a plant hormone for regulating stress responses
1.0000 | induces root tip closure in non-vital teeth
1.0000 | Cas Crop Genome Editing
1.0000 | Configuration data for kubernets work loads
1.0000 | popular rifle cartridge used for hunting and target shooting
1.0000 | a type of cancer treatment that uses drugs to boost the body's immune response
1.0000 | It is used by Kubernetes v1.35 native Pod certificate support, allowing workload certificates to be issued and rotated without relying entirely on external certificate controllers
1.0000 | It is used by Podman  native Pod certificate support, allowing workload certificates to be issued and rotated without relying entirely on external certificate controllers.


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit, scroll down!

In [28]:
model.save_pretrained("qwen_lora")  # Local saving
model.tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# model.tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

('qwen_lora/tokenizer_config.json',
 'qwen_lora/special_tokens_map.json',
 'qwen_lora/sentencepiece.bpe.model',
 'qwen_lora/added_tokens.json',
 'qwen_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [29]:
if False:
    from unsloth import FastSentenceTransformer
    model = FastSentenceTransformer.from_pretrained(
        "model",
        # for_inference = True
    )

### Saving to float16 for vLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [19]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf#locally)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [20]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen_finetune",)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>